# Functional Decoupling, Identity Construction, and the 2022 Semantic Rupture: NYT China Coverage (2020–2024)

This notebook produces every figure and quantitative result reported in **Sections 5–7** of the manuscript and the corresponding appendices. All analysis prose has been removed; full discussion appears in the manuscript. Each block below is labeled with the figure or table it produces and its location in the paper.

## Contents

| Section | Output | Paper location |
| --- | --- | --- |
| 0. Setup | Load full-text corpus | (setup) |
| 1.1 Lead-paragraph sentiment by desk | Figure 5 | §5.1 |
| 1.2 Full-text sentiment by desk | Figure 6 | §5.2 |
| 2.1–2.3 Adjectival framing (dependency parsing) | Figures 7, 8, 9 | §5.3 |
| 3. Regime vs. People sentiment disaggregation | Figure 10 | §5.4 |
| 4.1 Six-dimension emotion bar chart | Table C1 / Appendix C | §5.5 |
| 4.2 Fear and Anger time series | Figure 11 | §5.6 |
| 4.3 OLS regression on Fear and Anger | Regression table | §5.7 |
| 5. Distinctive keywords: Shanghai Lockdown and Spy Balloon | (text only) | §5.8 |
| 6.1 Identity register evolution | Figure 12 | §6.1 |
| 6.2 Crisis–Ideology see-saw | Figure 13 | §6.2 |
| 7.1 Word2Vec training and static projection | Figure 14 | §7.1 |
| 7.2 Year-by-year embedding trajectories | Figures 15, 16 | §7.2 |
| 7.3 Structural-break analysis on annual shifts | Figure 17 | §7.3 |
| 7.4 Pre/post-2022 nearest-neighbor analysis and qualitative retrieval | (text + neighbor tables) | §7.4 |

## Inputs and outputs

- **Input**: `China_Fulltext_Merged_2020-2024.csv` (full-text corpus produced upstream).
- **Outputs**: `NYT_Identity_Final_Results.csv` (static projection scores), `Yearly_Changes_Expanded.csv` (year-by-year embedding shifts), `Final_Geometry_Plot.png` (publication figure).

## Dependencies

`pandas`, `numpy`, `matplotlib`, `seaborn`, `scipy`, `statsmodels`, `nltk` (with `punkt`), `scikit-learn`, `spacy` (with `en_core_web_sm`), `transformers` (`distilbert-base-uncased-finetuned-sst-2-english`, `j-hartmann/emotion-english-distilroberta-base`), `gensim`, `torch`.

## Note on outputs and reproducibility

Cell outputs have been stripped to keep the file lightweight; running all cells reproduces the figures end-to-end. Two analyses involve stochastic components: the sentence-level disaggregation in §5.4 draws a random 20,000-sentence sample, and the Word2Vec models in §7 are trained with multi-core parallelism. Re-running these cells produces results within ±0.01 of the thesis numbers but not exact bitwise replicas. The figures and exact statistics reported in the manuscript correspond to a specific run preserved in the author's working directory.

## 0. Setup

Load the merged full-text corpus and inspect the news-desk distribution.

**0.1 Load the full-text corpus.**

In [ ]:
import pandas as pd
df_text = pd.read_csv('China_Fulltext_Merged_2020-2024.csv')

**0.2 Inspect news-desk distribution.**

In [ ]:
df_text['news_desk'].value_counts()

## 1. Sentiment Analysis (§5.1–5.2)

Two parallel sentiment measurements: lead paragraphs (the editorial "hook") and full-text via 250-word chunked aggregation. Both use DistilBERT (`distilbert-base-uncased-finetuned-sst-2-english`).


### 1.1 Figure 5 — Lead-paragraph sentiment by desk (§5.1)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from transformers import pipeline
from tqdm.auto import tqdm  # progress bar

# ==========================================
# 1. Load Data & Clean
# ==========================================
df_text['full_text'] = df_text['full_text'].astype(str)

# ==========================================
# 2. Refined Categorization (Based on your Screenshot)
# ==========================================
def categorize_section_precise(desk):
    if not isinstance(desk, str):
        return 'Other'
    
    # Convert to lowercase for matching
    desk = desk.strip() 
    
    # Define precise desk sets
    econ_group = {
        'Business', 'Business Day', 'SundayBusiness', 'Technology', 'Science'
    }
    
    poli_group = {
        'Foreign', 'Washington', 'World', 'Politics', 'National', 
        'Editorial', 'Investigative', 'Briefing'
    }
    
    if desk in econ_group:
        return 'Economic/Rational'
    elif desk in poli_group:
        return 'Political/Hostile'
    else:
        return 'Other'

# Apply categorization
print("Applying categorization...")
df_text['functional_frame'] = df_text['news_desk'].apply(categorize_section_precise)

# Filter data: keep only the two functional frames
df_analysis = df_text[df_text['functional_frame'].isin(['Economic/Rational', 'Political/Hostile'])].copy()

print("--- Data Distribution ---")
print(df_analysis['functional_frame'].value_counts())

# ==========================================
# 3. High-Accuracy Sentiment Analysis (Transformer)
# ==========================================
# Initialize Hugging Face pipeline
# DistilBERT: fast and substantially more accurate than VADER
# Output: NEGATIVE / POSITIVE label plus confidence score
print("\nLoading Transformer Model (this may take a minute)...")
sentiment_pipeline = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english", truncation=True, max_length=512)

def get_transformer_score(text):
    # Truncate to first 512 tokens (Transformer limit); the lead sets the tone
    # Map confidence to a signed score: -1 (Neg) to +1 (Pos)
    try:
        result = sentiment_pipeline(text[:2000])[0]  # pass a leading slice of the text
        label = result['label']
        score = result['score']
        
        # distilbert-sst-2 labels are NEGATIVE / POSITIVE
        if label == 'NEGATIVE':
            return -score  # negative score
        else:
            return score  # positive score
    except Exception as e:
        return 0

print(f"\nAnalyzing sentiment for {len(df_analysis)} articles using Transformer...")
print("Please wait, this might take 10-20 minutes depending on your CPU...")

# Use tqdm progress bar
tqdm.pandas()
df_analysis['sentiment_score'] = df_analysis['full_text'].progress_apply(get_transformer_score)

# ==========================================
# 4. Statistical Test (T-test)
# ==========================================
group_econ = df_analysis[df_analysis['functional_frame'] == 'Economic/Rational']['sentiment_score']
group_pol = df_analysis[df_analysis['functional_frame'] == 'Political/Hostile']['sentiment_score']

t_stat, p_val = stats.ttest_ind(group_econ, group_pol, equal_var=False)

print("\n--- Statistical Results (Transformer) ---")
print(f"Economic Group Mean: {group_econ.mean():.4f}")
print(f"Political Group Mean: {group_pol.mean():.4f}")
print(f"Difference: {group_econ.mean() - group_pol.mean():.4f}")
print(f"P-value: {p_val:.4e}")

# ==========================================
# 5. Visualization
# ==========================================
plt.figure(figsize=(10, 6))
sns.set_style("whitegrid")

# Violin Plot
sns.violinplot(x='functional_frame', y='sentiment_score', data=df_analysis, 
               palette=['#ffcc00', '#1f77b4'], alpha=0.6, inner=None)
# Box Plot inside
sns.boxplot(x='functional_frame', y='sentiment_score', data=df_analysis, 
            width=0.1, boxprops={'zorder': 2, 'facecolor':'white'}, showfliers=False)

plt.title('The "Schizophrenia" Test: Sentiment Divergence (Transformer Model)', fontsize=14, fontweight='bold')
plt.xlabel('Narrative Context')
plt.ylabel('Sentiment Score (-1=Neg, +1=Pos)')
plt.axhline(0, color='red', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

### 1.2 Figure 6 — Full-text sentiment by desk via chunked aggregation (§5.2)

In [ ]:
import pandas as pd
import numpy as np
import torch
from transformers import pipeline
from tqdm.auto import tqdm

# ==========================================
# 0. Setup (Device & Pipeline)
# ==========================================
try:
    if torch.backends.mps.is_available():
        device = "mps"
        print("Using MPS (Mac GPU).")
    elif torch.cuda.is_available():
        device = 0
        print("Using CUDA (Nvidia GPU).")
    else:
        device = -1
        print("Using CPU.")
except:
    device = -1

# --- Critical change ---
# Add truncation=True and max_length=512
# This acts as a safety fuse: we chunk manually at 250 words, but if any chunk overshoots, the model truncates instead of erroring.
sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    device=device,
    truncation=True,   # <--- required
    max_length=512    # <--- required
)

# ==========================================
# 1. Define Chunking Function
# ==========================================
def get_full_text_sentiment(text, chunk_size=250):  # 250 fixed
    if not isinstance(text, str) or len(text.strip()) == 0:
        return 0.0
        
    words = text.split()
    
    # Case A: short article
    if len(words) <= chunk_size:
        try:
            # truncation=True above makes this safe even if slightly over the limit
            res = sentiment_pipeline(" ".join(words))[0]
            score = res['score']
            if res['label'] == 'NEGATIVE':
                score = -score
            return score
        except:
            return 0.0
            
    # Case B: long article — chunked
    chunks = []
    for i in range(0, len(words), chunk_size):
        chunk = " ".join(words[i : i + chunk_size])
        chunks.append(chunk)
    
    try:
        # Batched prediction
        # Even if a chunk is unlucky and tokenizes to 520 tokens,
        # the pipeline now auto-truncates to 512 instead of crashing.
        results = sentiment_pipeline(chunks)
        
        scores = []
        for res in results:
            s = res['score']
            if res['label'] == 'NEGATIVE':
                s = -s
            scores.append(s)
            
        return np.mean(scores)
        
    except Exception as e:
        # print(f"Error: {e}")  # uncomment for debugging
        return 0.0

# ==========================================
# 2. Execution
# ==========================================
# Requires df_text in scope
# df_text = ... 

df_analysis = df_text[df_text['functional_frame'].isin(['Economic/Rational', 'Political/Hostile'])].copy()

print(f"Analyzing FULL TEXT for {len(df_analysis)} articles using Chunking Strategy...")
tqdm.pandas()
df_analysis['sentiment_score_full'] = df_analysis['full_text'].progress_apply(get_full_text_sentiment)

# ==========================================
# 3. Stats & Vis
# ==========================================
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

group_econ = df_analysis[df_analysis['functional_frame'] == 'Economic/Rational']['sentiment_score_full']
group_pol = df_analysis[df_analysis['functional_frame'] == 'Political/Hostile']['sentiment_score_full']

t_stat, p_val = stats.ttest_ind(group_econ, group_pol, equal_var=False)

print("\n--- FULL TEXT Statistical Results ---")
print(f"Economic Group Mean: {group_econ.mean():.4f}")
print(f"Political Group Mean: {group_pol.mean():.4f}")
print(f"Difference: {group_econ.mean() - group_pol.mean():.4f}")
print(f"P-value: {p_val:.4e}")

plt.figure(figsize=(10, 6))
sns.set_style("whitegrid")
sns.violinplot(x='functional_frame', y='sentiment_score_full', data=df_analysis, 
               palette=['#ffcc00', '#1f77b4'], alpha=0.6, inner="box")
plt.title('The "Schizophrenia" Test: Full Text Sentiment Divergence', fontsize=14, fontweight='bold')
plt.xlabel('Narrative Context')
plt.ylabel('Sentiment Score (Average of Chunks)')
plt.axhline(0, color='red', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## 2. Adjectival Framing via Dependency Parsing (§5.3)

Adjectives modifying target nouns are extracted using spaCy's dependency parser. Two cases are captured: prenominal modifiers (`amod`, e.g., "weak economy") and predicative modifiers (e.g., "the economy is weak"). Output drives Figures 7, 8, and 9.


### 2.1 Adjective extraction: China political vs. economic targets

In [ ]:
import spacy
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from tqdm.auto import tqdm

# ==========================================
# 0. Setup & GPU Check
# ==========================================
# Try to enable GPU
is_gpu = spacy.prefer_gpu()
if is_gpu:
    print("spaCy is using GPU.")
else:
    print("spaCy is using CPU.")

try:
    nlp = spacy.load("en_core_web_sm")
except:
    from spacy.cli import download
    download("en_core_web_sm")
    nlp = spacy.load("en_core_web_sm")

# ==========================================
# 1. Define Syntax Extraction Logic (FIXED)
# ==========================================
def extract_adjectives_from_doc(doc, target_nouns_set):
    adjectives = []
    
    for token in doc:
        # --- FIX 1: force lowercase comparison ---
        # Check whether the token is in the target noun set
        if token.text.lower() in target_nouns_set:
            
            # Case A: prenominal modifier (the *weak* economy)
            for child in token.children:
                if child.dep_ == 'amod' and child.pos_ == 'ADJ':
                    # Use lemma if available, otherwise surface form
                    adj = child.lemma_ if child.lemma_ else child.text
                    adjectives.append(adj.lower())
            
            # Case B: predicative modifier (the economy is *weak*)
            if token.dep_ == 'nsubj': 
                verb = token.head
                for child in verb.children:
                    if (child.dep_ == 'acomp' or child.dep_ == 'attr') and child.pos_ == 'ADJ':
                        adj = child.lemma_ if child.lemma_ else child.text
                        adjectives.append(adj.lower())
                        
    return adjectives

# ==========================================
# 2. Define Targets (Lowercase)
# ==========================================
# Ensure lowercase
targets_pol = {'china', 'beijing', 'party', 'ccp', 'government', 'nation', 'xi', 'president'}
targets_econ = {'economy', 'market', 'growth', 'sector', 'industry', 'trade', 'debt', 'recovery', 'business', 'supply', 'chain'}

# ==========================================
# 3. Execution (Batch Processing)
# ==========================================
print("Extracting adjectives using nlp.pipe...")

# Prepare data
df_econ_desk = df_analysis[df_analysis['functional_frame'] == 'Economic/Rational'].copy()
df_pol_desk = df_analysis[df_analysis['functional_frame'] == 'Political/Hostile'].copy()

def process_batch(df, desc):
    china_adjs = []
    econ_adjs = []
    
    texts = df['full_text'].astype(str).tolist()
    
    # --- FIX 2: do NOT disable 'lemmatizer' — we need it to recover adjective base forms ---
    # Only disable 'ner' and 'textcat' for speed
    n_proc = 1 if is_gpu else -1 
    
    for doc in tqdm(nlp.pipe(texts, batch_size=100, disable=["ner", "textcat"], n_process=n_proc), total=len(texts), desc=desc):
        china_adjs.extend(extract_adjectives_from_doc(doc, targets_pol))
        econ_adjs.extend(extract_adjectives_from_doc(doc, targets_econ))
        
    return china_adjs, econ_adjs

# Execute
econ_desk_CHINA, econ_desk_ECON = process_batch(df_econ_desk, "Processing Business Desk")
pol_desk_CHINA, pol_desk_ECON = process_batch(df_pol_desk, "Processing Political Desk")



### 2.2 Stopword list for adjective filtering

In [ ]:
# ==========================================
# 5. Final aggressive stoplist
# ==========================================

# 1. Generic quantifiers and adjectives
base_stops = {
    'many', 'much', 'more', 'most', 'other', 'same', 'such', 'new', 'own', 'first', 'last', 
    'top', 'major', 'big', 'large', 'good', 'bad', 'great', 'high', 'low', 'second', 'third',
    'hard', 'early', 'late', 'likely', 'possible', 'able', 'full', 'small', 'recent', 'long', 
    'little', 'huge', 'vast', 'short', 'broad', 'wide', 'single', 'several', 'various', 
    'main', 'general', 'real', 'whole', 'entire', 'important', 'significant', 'key'
}

# 2. Administrative / geographic / directional terms (drop e.g. municipal, southeastern)
geo_stops = {
    'central', 'local', 'federal', 'national', 'global', 'international', 'domestic',
    'foreign', 'western', 'eastern', 'southern', 'northern', 'southeastern', 'southwestern',
    'northeastern', 'northwestern', 'municipal', 'regional', 'urban', 'rural', 'overseas',
    'internal', 'external','northeast', 'north', 'south', 'east', 'west'
}

# 3. Temporal / sequence / state terms (drop e.g. former, willing)
time_stops = {
    'former', 'senior', 'current', 'past', 'next', 'previous', 'future', 'willing', 
    'ready', 'likely', 'unlikely', 'expected', 'potential', 'pro', 'anti'
}

# 4. Nationality and place-derived adjectives (drop e.g. german, russian, chinese)
nation_stops = {
    'chinese', 'american', 'british', 'european', 'russian', 'german', 'japanese', 
    'taiwanese', 'australian', 'canadian', 'indian', 'french', 'ukrainian', 'asian', 
    'african', 'latin', 'korean'
}

# 5. Industry / context-specific modifiers (drop e.g. solar, free, electric, digital)
context_stops = {
    'economic', 'political', 'social', 'financial', 'industrial', 'technological',
    'solar', 'electric', 'digital', 'green', 'clean', 'public', 'private', 'free', 
    'open', 'executive', 'communist', 'democratic', 'republican', 'military', 'civil',
    'legal', 'official', 'traditional', 'modern'
}

# Merge all stopwords
final_stop_adjs = base_stops | geo_stops | time_stops | nation_stops | context_stops

# ==========================================
# 6. Re-Filter & Re-Visualize
# ==========================================
def filter_and_count_aggressive(adjs_list):
    # Filter logic:
    # 1. Not in the stopword list
    # 2. Length > 3 (drop short tokens like 'pro')
    clean = [
        a.lower() for a in adjs_list 
        if a.lower() not in final_stop_adjs 
        and len(a) > 3
    ]
    return Counter(clean).most_common(10)

# Rebuild plotting data
data_map_final = {
    'Business Desk describing CHINA': filter_and_count_aggressive(econ_desk_CHINA),
    'Political Desk describing CHINA': filter_and_count_aggressive(pol_desk_CHINA),
    'Business Desk describing ECONOMY': filter_and_count_aggressive(econ_desk_ECON),
    'Political Desk describing ECONOMY': filter_and_count_aggressive(pol_desk_ECON)
}

# Plot
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()
colors = ['#ffcc00', '#1f77b4', '#ffcc00', '#1f77b4'] 

for i, (title, top_data) in enumerate(data_map_final.items()):
    ax = axes[i]
    if not top_data:
        ax.text(0.5, 0.5, "Not enough data after filtering", ha='center')
        continue
        
    words = [x[0] for x in top_data][::-1]
    counts = [x[1] for x in top_data][::-1]
    
    ax.barh(words, counts, color=colors[i], edgecolor='black', alpha=0.8)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('Frequency')

plt.suptitle('The "Adjective Framing": Qualitative Nuance', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

### 2.3 Adjective extraction: U.S. targets across desks

In [ ]:
import spacy
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from tqdm.auto import tqdm

# ==========================================
# 0. Setup & GPU Check
# ==========================================
is_gpu = spacy.prefer_gpu()
if is_gpu:
    print("spaCy is using GPU.")
else:
    print("spaCy is using CPU.")

try:
    nlp = spacy.load("en_core_web_sm")
except:
    from spacy.cli import download
    download("en_core_web_sm")
    nlp = spacy.load("en_core_web_sm")

# ==========================================
# 1. Syntax extraction logic (same as previous cell)
# ==========================================
def extract_adjectives_from_doc(doc, target_nouns_set):
    adjectives = []
    for token in doc:
        if token.text.lower() in target_nouns_set:
            # Case A: prenominal modifier (the *strong* economy)
            for child in token.children:
                if child.dep_ == 'amod' and child.pos_ == 'ADJ':
                    adj = child.lemma_ if child.lemma_ else child.text
                    adjectives.append(adj.lower())
            # Case B: predicative modifier (the economy is *strong*)
            if token.dep_ == 'nsubj': 
                verb = token.head
                for child in verb.children:
                    if (child.dep_ == 'acomp' or child.dep_ == 'attr') and child.pos_ == 'ADJ':
                        adj = child.lemma_ if child.lemma_ else child.text
                        adjectives.append(adj.lower())
    return adjectives

# ==========================================
# 2. Define Targets (PURIFIED) ✨
# ==========================================
# Removed: 'government', 'nation', 'country', 'administration', 'president'
# Keep only proper nouns unambiguously denoting the U.S.

targets_us_pure = {
    # Country names
    'america', 'usa', 'u.s.', 'us', 'washington', 
    # Leaders / institutions
    'biden', 'trump', 'whitehouse', 'pentagon', 
    # Legislative bodies / parties
    'congress', 'senate', 'democrats', 'republicans', 'democrat', 'republican'
}

# Economic vocabulary unchanged
targets_econ = {
    'economy', 'market', 'growth', 'sector', 'industry', 
    'trade', 'debt', 'recovery', 'business', 'supply', 'chain',
    'inflation', 'rates', 'fed', 'recession'
}

# ==========================================
# 3. Execution (Batch Processing)
# ==========================================
print("Extracting adjectives for US (Purified) & Economy...")

# Assumes df_econ_desk and df_pol_desk are in memory
# df_econ_desk = df_analysis[df_analysis['functional_frame'] == 'Economic/Rational'].copy()
# df_pol_desk = df_analysis[df_analysis['functional_frame'] == 'Political/Hostile'].copy()

def process_batch(df, desc):
    us_adjs = []
    econ_adjs = []
    
    texts = df['full_text'].astype(str).tolist()
    n_proc = 1 if is_gpu else -1 
    
    # Batched processing with nlp.pipe
    for doc in tqdm(nlp.pipe(texts, batch_size=100, disable=["ner", "textcat"], n_process=n_proc), total=len(texts), desc=desc):
        us_adjs.extend(extract_adjectives_from_doc(doc, targets_us_pure))  # use the cleaned target list
        econ_adjs.extend(extract_adjectives_from_doc(doc, targets_econ))
        
    return us_adjs, econ_adjs

# Execute extraction
econ_desk_US, econ_desk_ECON = process_batch(df_econ_desk, "Processing Business Desk")
pol_desk_US, pol_desk_ECON = process_batch(df_pol_desk, "Processing Political Desk")

# ==========================================
# 4. The "Nuclear" Stoplist (Unchanged)
# ==========================================
# Stopword list kept as-is — effective at filtering uninformative adjectives
base_stops = {
    'many', 'much', 'more', 'most', 'other', 'same', 'such', 'new', 'own', 'first', 'last', 
    'top', 'major', 'big', 'large', 'good', 'bad', 'great', 'high', 'low', 'second', 'third',
    'hard', 'early', 'late', 'likely', 'possible', 'able', 'full', 'small', 'recent', 'long', 
    'little', 'huge', 'vast', 'short', 'broad', 'wide', 'single', 'several', 'various', 
    'main', 'general', 'real', 'whole', 'entire', 'important', 'significant', 'key'
}
geo_stops = {
    'central', 'local', 'federal', 'national', 'global', 'international', 'domestic',
    'foreign', 'western', 'eastern', 'southern', 'northern', 'southeastern', 'southwestern',
    'northeastern', 'northwestern', 'municipal', 'regional', 'urban', 'rural', 'overseas',
    'internal', 'external','northeast', 'north', 'south', 'east', 'west'
}
time_stops = {
    'former', 'senior', 'current', 'past', 'next', 'previous', 'future', 'willing', 
    'ready', 'likely', 'unlikely', 'expected', 'potential', 'pro', 'anti'
}
nation_stops = {
    'chinese', 'american', 'british', 'european', 'russian', 'german', 'japanese', 
    'taiwanese', 'australian', 'canadian', 'indian', 'french', 'ukrainian', 'asian', 
    'african', 'latin', 'korean'
}
context_stops = {
    'economic', 'political', 'social', 'financial', 'industrial', 'technological',
    'solar', 'electric', 'digital', 'green', 'clean', 'public', 'private', 'free', 
    'open', 'executive', 'communist', 'democratic', 'republican', 'military', 'civil',
    'legal', 'official', 'traditional', 'modern'
}
final_stop_adjs = base_stops | geo_stops | time_stops | nation_stops | context_stops

# ==========================================
# 5. Re-Filter & Re-Visualize
# ==========================================
def filter_and_count_aggressive(adjs_list):
    clean = [
        a.lower() for a in adjs_list 
        if a.lower() not in final_stop_adjs 
        and len(a) > 3
    ]
    return Counter(clean).most_common(10)

# Update data mapping
data_map_final_us = {
    'Business Desk describing USA': filter_and_count_aggressive(econ_desk_US),
    'Political Desk describing USA': filter_and_count_aggressive(pol_desk_US),
    'Business Desk describing ECONOMY': filter_and_count_aggressive(econ_desk_ECON),
    'Political Desk describing ECONOMY': filter_and_count_aggressive(pol_desk_ECON)
}

# Plot
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()
colors = ['#d62728', '#1f77b4', '#d62728', '#1f77b4']  # red = U.S.

for i, (title, top_data) in enumerate(data_map_final_us.items()):
    ax = axes[i]
    if not top_data:
        ax.text(0.5, 0.5, "Not enough data after filtering", ha='center')
        continue
        
    words = [x[0] for x in top_data][::-1]
    counts = [x[1] for x in top_data][::-1]
    
    ax.barh(words, counts, color=colors[i], edgecolor='black', alpha=0.8)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('Frequency')

plt.suptitle('Adjective Framing: How NYT Described the USA (Purified Targets)', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

### 2.4 Figures 7, 8, 9 — Adjectival framing across desks

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

# ==========================================
# 1. Final blacklist (built from prior outputs)
# ==========================================
# Manually exclude all category, directional, and nationality terms
scorched_earth_list = {
    # Noise terms observed in prior runs
    'national', 'congressional', 'private', 'international', 'american', 
    'domestic', 'european', 'local', 'automotive', 'ranking', 'other', 
    'many', 'first', 'short', 'sino', 'german', 'entire', 'overall', 
    'broad', 'vast', 'huge', 'rapid', 'senior', 'junior', 'public',
    'federal', 'industrial', 'financial', 'corporate', 'global',
    'western', 'eastern', 'southern', 'northern', 'foreign', 'civil',
    'military', 'legal', 'social', 'political', 'official', 'executive',
    'top', 'major', 'key', 'main', 'primary', 'significant', 'important',
    'likely', 'able', 'ready', 'possible', 'potential', 'expected',
    'new', 'old', 'current', 'recent', 'past', 'future', 'modern',
    'dependent', 'reluctant', 'short', 'long', 'small', 'large'
}

# ==========================================
# 2. Suffix-based filter
# ==========================================
def is_evaluative(word):
    w = word.lower()
    
    # Rule 1: drop if in blacklist
    if w in scorched_earth_list:
        return False
        
    # Rule 2: drop -al endings (mostly category terms: national, federal, structural)
    # Whitelist exceptions: critical, radical, central, liberal
    whitelist_al = {'critical', 'radical', 'central', 'liberal', 'vital', 'loyal'}
    if w.endswith('al') and w not in whitelist_al:
        return False
        
    # Rule 3: drop -an / -ese / -ish endings (nationality terms)
    if w.endswith('an') or w.endswith('ese') or w.endswith('ish'):
        return False
        
    return True

# ==========================================
# 3. Apply cleaning
# ==========================================
def final_clean(adjs_list):
    clean = []
    for a in adjs_list:
        if is_evaluative(a) and len(a) > 3:
            clean.append(a.lower())
    return Counter(clean).most_common(8)  # top 8 only — concentrated signal

# Rebuild from prior extraction variables
data_map_final = {
    'Business Desk on USA': final_clean(econ_desk_US),
    'Political Desk on USA': final_clean(pol_desk_US),
    'Business Desk on ECONOMY': final_clean(econ_desk_ECON),
    'Political Desk on ECONOMY': final_clean(pol_desk_ECON)
}

# ==========================================
# 4. Plot
# ==========================================
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()
colors = ['#d62728', '#1f77b4', '#d62728', '#1f77b4'] 

print("--- 📝 The Final List (Diamond in the Rough) ---")

for i, (title, top_data) in enumerate(data_map_final.items()):
    ax = axes[i]
    
    # Print plain-text version
    words = [x[0] for x in top_data]
    print(f"\n[{title}]: {words}")
    
    if not top_data:
        ax.text(0.5, 0.5, "Too much cleaning?", ha='center')
        continue
        
    counts = [x[1] for x in top_data]
    
    # Reverse for horizontal bar plotting
    ax.barh(words[::-1], counts[::-1], color=colors[i], edgecolor='black', alpha=0.8)
    ax.set_title(title, fontsize=14, fontweight='bold')

plt.suptitle('Constructing the Self: The "Character" of the US (Final)', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

## 3. Sentence-Level Disaggregation (§5.4)

Sentences mentioning regime/state actors are scored separately from sentences mentioning the Chinese people / civil society. Each group is sampled to 20,000 sentences for sentiment classification with the same DistilBERT model.


### 3.1 Figure 10 — Regime/Party vs. People/Civil Society sentence sentiment

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
import nltk
from transformers import pipeline
import torch

# ==========================================
# 0. Setup
# ==========================================
# Ensure sentence tokenizer is available
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt')

# Reload pipeline as a safety measure
try:
    if torch.backends.mps.is_available():
        device = "mps"
    elif torch.cuda.is_available():
        device = 0
    else:
        device = -1
except:
    device = -1

# Enable truncation to prevent errors on long sentences
sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    device=device,
    truncation=True,
    max_length=512
)

# ==========================================
# 1. Define Keywords for Filtering
# ==========================================
# Group A: Regime / State / Party
keywords_regime = {
    'ccp', 'communist party', 'party', 'government', 'beijing', 'xi', 'jinping', 
    'regime', 'authorities', 'official', 'state', 'ministry', 'pla', 'military'
}

# Group B: People / Civil Society
keywords_people = {
    'chinese people', 'citizens', 'residents', 'workers', 'students', 'families', 
    'population', 'public', 'society', 'women', 'men', 'children', 'villagers', 
    'consumers', 'middle class', 'youth'
}

# ==========================================
# 2. Extract & Classify Sentences
# ==========================================
print("Splitting articles into sentences and filtering... (This prepares the data)")

regime_sentences = []
people_sentences = []

# Iterate over all articles
# Use nltk.sent_tokenize for sentence segmentation
for text in tqdm(df_text['full_text'].astype(str)):
    sentences = nltk.sent_tokenize(text)
    
    for sent in sentences:
        sent_lower = sent.lower()
        
        # Check for keyword presence
        has_regime = any(k in sent_lower for k in keywords_regime)
        has_people = any(k in sent_lower for k in keywords_people)
        
        # Mutually exclusive classification: keep sentences that target only regime or only people
        # Mixed sentences (e.g. "The CCP oppressed the people") are dropped
        if has_regime and not has_people:
            regime_sentences.append(sent)
        elif has_people and not has_regime:
            people_sentences.append(sent)

print(f"\nFound {len(regime_sentences)} sentences about REGIME.")
print(f"Found {len(people_sentences)} sentences about PEOPLE.")

# ==========================================
# 3. Calculate Sentiment (Batch Processing)
# ==========================================
print("\nCalculating sentiment for sentences... (This takes time)")

def get_batch_sentiment(sentences, batch_size=64):
    scores = []
    # Batched processing
    for i in tqdm(range(0, len(sentences), batch_size)):
        batch = sentences[i : i + batch_size]
        try:
            results = sentiment_pipeline(batch)
            for res in results:
                score = res['score']
                if res['label'] == 'NEGATIVE':
                    score = -score
                scores.append(score)
        except Exception as e:
            # On error, fill with 0
            scores.extend([0.0] * len(batch))
    return scores

# Sample for tractability if the corpus is large
# With GPU, tens of thousands of sentences run in minutes; here we cap at 20,000 per group
import random

# Cap sample size at 20,000 per group (sufficient for statistical significance)
max_samples = 20000
if len(regime_sentences) > max_samples:
    regime_sentences = random.sample(regime_sentences, max_samples)
if len(people_sentences) > max_samples:
    people_sentences = random.sample(people_sentences, max_samples)

print(f"Running sentiment on {len(regime_sentences)} Regime sentences...")
regime_scores = get_batch_sentiment(regime_sentences)

print(f"Running sentiment on {len(people_sentences)} People sentences...")
people_scores = get_batch_sentiment(people_sentences)

# ==========================================
# 4. Visualization & Stats
# ==========================================
# Build plotting data
df_viz = pd.DataFrame({
    'Sentiment': regime_scores + people_scores,
    'Target': ['Regime/Party'] * len(regime_scores) + ['People/Civil'] * len(people_scores)
})

# T-test
from scipy import stats
t_stat, p_val = stats.ttest_ind(regime_scores, people_scores, equal_var=False)

print("\n--- The 'Party vs. People' Test Results ---")
print(f"Regime Mean: {sum(regime_scores)/len(regime_scores):.4f}")
print(f"People Mean: {sum(people_scores)/len(people_scores):.4f}")
print(f"Difference: {(sum(people_scores)/len(people_scores)) - (sum(regime_scores)/len(regime_scores)):.4f}")
print(f"P-value: {p_val:.4e}")

# Plot
plt.figure(figsize=(10, 6))
sns.set_style("whitegrid")
sns.violinplot(x='Target', y='Sentiment', data=df_viz, palette=['#d62728', '#2ca02c'], inner="box", alpha=0.7)
plt.title('The "Party vs. People" Test: Sentiment Disaggregation', fontsize=14, fontweight='bold')
plt.xlabel('Target Entity')
plt.ylabel('Sentence Sentiment Score')
plt.axhline(0, color='gray', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

## 4. Six-Dimension Emotion and Regression (§5.5–5.7)

Emotion classification uses `j-hartmann/emotion-english-distilroberta-base` with 250-word chunking and chunk-averaged probabilities. The regression model isolates the institutional desk effect on Fear and Anger after controlling for topic and time.


### 4.1 Table C1 (Appendix C) — Full-text six-dimension emotion comparison (§5.5)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from transformers import pipeline
from tqdm.auto import tqdm
import torch

# ==========================================
# 0. Setup (Load Model)
# ==========================================
try:
    if torch.backends.mps.is_available():
        device = "mps"
        print("Using MPS (Mac GPU).")
    elif torch.cuda.is_available():
        device = 0
        print("Using CUDA (Nvidia GPU).")
    else:
        device = -1
        print("Using CPU.")
except:
    device = -1

print("Loading Emotion Model...")

# FIX 1: Use top_k=None instead of the deprecated return_all_scores=True
# Remove truncation and max_length from here (they belong in the call)
emotion_pipeline = pipeline(
    "text-classification", 
    model="j-hartmann/emotion-english-distilroberta-base", 
    top_k=None, 
    device=device
)

# ==========================================
# 1. Define Chunking Function
# ==========================================
def get_full_text_emotions(text, chunk_size=250):
    if not isinstance(text, str) or len(text.strip()) == 0:
        return None

    words = text.split()
    chunks = [" ".join(words[i : i + chunk_size]) for i in range(0, len(words), chunk_size)]
            
    emotion_sums = {
        'anger': 0.0, 'disgust': 0.0, 'fear': 0.0, 'joy': 0.0, 
        'neutral': 0.0, 'sadness': 0.0, 'surprise': 0.0
    }
    count = 0
    
    try:
        # FIX 2: Apply truncation and max_length DURING the inference call
        results = emotion_pipeline(chunks, batch_size=8, truncation=True, max_length=512)
        
        for chunk_res in results:
            for item in chunk_res:
                emotion_sums[item['label']] += item['score']
            count += 1
            
        if count > 0:
            return {k: v / count for k, v in emotion_sums.items()}
        return None
            
    except Exception as e:
        # FIX 3: Stop swallowing errors silently! Print them out.
        print(f"Pipeline failed on a text: {e}")
        return None

# ==========================================
# 2. Execution
# ==========================================
df_analysis = df_text[df_text['functional_frame'].isin(['Economic/Rational', 'Political/Hostile'])].copy()

print(f"Analyzing FULL TEXT Emotions for {len(df_analysis)} articles...")
tqdm.pandas()
df_analysis['emotion_dict'] = df_analysis['full_text'].progress_apply(get_full_text_emotions)

# ==========================================
# 3. Data Formatting & Aggregation
# ==========================================
# FIX 4: Add .tolist() so json_normalize gets a clean Python list, not a Series
emotion_df = pd.json_normalize(df_analysis['emotion_dict'].tolist())
emotion_df.index = df_analysis.index 

df_final = pd.concat([df_analysis[['functional_frame']], emotion_df], axis=1)

result_means = df_final.groupby('functional_frame').mean()
print("\n--- Full Text Emotion Results ---")
print(result_means)

emotions = ['anger', 'disgust', 'fear', 'joy', 'sadness', 'surprise']  # exclude 'neutral'

econ_vals = [result_means.loc['Economic/Rational', e] for e in emotions]
poli_vals = [result_means.loc['Political/Hostile', e] for e in emotions]

x = np.arange(len(emotions))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 7))
rects1 = ax.bar(x - width/2, econ_vals, width, label='Business Desk', color='#ffcc00', alpha=0.9, edgecolor='black')
rects2 = ax.bar(x + width/2, poli_vals, width, label='Political Desk', color='#1f77b4', alpha=0.9, edgecolor='black')

ax.set_ylabel('Intensity (Average Probability)', fontsize=12)
ax.set_title('Full-Text Emotion Analysis: The "True" Emotional Baseline', fontsize=16, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(emotions, fontsize=12)
ax.legend(fontsize=12)
ax.grid(axis='y', linestyle='--', alpha=0.5)

# Annotate numeric differences
def autolabel(rects):
    for rect in rects:
        height = rect.get_height()
        ax.annotate(f'{height:.3f}',
                    xy=(rect.get_x() + rect.get_width() / 2, height),
                    xytext=(0, 3),  # 3 points vertical offset
                    textcoords="offset points",
                    ha='center', va='bottom', fontsize=9)

autolabel(rects1)
autolabel(rects2)

plt.tight_layout()
plt.show()

# Print exact differences for citation in the manuscript
print("\n--- Detailed Differences (Business - Politics) ---")
for e in emotions:
    diff = result_means.loc['Economic/Rational', e] - result_means.loc['Political/Hostile', e]
    print(f"{e:<10}: {diff:+.4f}")

### 4.2 Figure 11 — Fear and Anger time series across desks (§5.6)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ==========================================
# 1. Expand Emotion Dictionary
# ==========================================
# Expand emotion_dict (dict column) into individual columns (anger, fear, etc.)
# Skip if already expanded
if 'fear' not in df_analysis.columns and 'emotion_dict' in df_analysis.columns:
    print("Expanding emotion dictionary into columns...")
    emotion_df = pd.json_normalize(df_analysis['emotion_dict'])
    # Align index
    emotion_df.index = df_analysis.index
    # Concatenate
    df_analysis = pd.concat([df_analysis, emotion_df], axis=1)

# ==========================================
# 2. Prepare Time Data
# ==========================================
# Ensure datetime parsing
if 'pub_date' in df_analysis.columns:
    df_analysis['date'] = pd.to_datetime(df_analysis['pub_date'])
elif 'date' in df_analysis.columns:
    df_analysis['date'] = pd.to_datetime(df_analysis['date'])

# Aggregate by quarter (Q) or month (M); use 'Q' for smoother curves
df_analysis['time_period'] = df_analysis['date'].dt.to_period('Q') 

# ==========================================
# 3. Aggregate & Plot: The "Fear" Narrative
# ==========================================
# Focus on 'fear' (business anxiety) and 'anger' (political hostility)
emotions_to_plot = ['fear', 'anger', 'sadness']

# Aggregate
emotion_trend = df_analysis.groupby(['time_period', 'functional_frame'])[emotions_to_plot].mean().reset_index()
emotion_trend['date'] = emotion_trend['time_period'].dt.to_timestamp()

# Plot setup
fig, axes = plt.subplots(2, 1, figsize=(14, 12), sharex=True)
sns.set_style("whitegrid")

# --- Plot 1: Fear timeline ---
sns.lineplot(
    ax=axes[0],
    x='date', y='fear', hue='functional_frame', data=emotion_trend,
    palette={'Economic/Rational': '#ffcc00', 'Political/Hostile': '#1f77b4'},
    linewidth=3, marker='o'
)
axes[0].set_title('Narrative of Anxiety: Evolution of "Fear" (2020-2024)', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Fear Intensity')
axes[0].legend(loc='upper right')

# --- Plot 2: Anger timeline ---
sns.lineplot(
    ax=axes[1],
    x='date', y='anger', hue='functional_frame', data=emotion_trend,
    palette={'Economic/Rational': '#ffcc00', 'Political/Hostile': '#1f77b4'},
    linewidth=3, marker='o'
)
axes[1].set_title('Narrative of Hostility: Evolution of "Anger" (2020-2024)', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Anger Intensity')
axes[1].set_xlabel('Date')

# Annotate key events
events = {
    '2020-01': 'Covid Start',
    '2021-07': 'Tech Crackdown',
    '2022-04': 'Shanghai Lockdown',
    '2023-02': 'Spy Balloon'
}

for ax in axes:
    for date_str, event_name in events.items():
        date_obj = pd.to_datetime(date_str)
        ax.axvline(date_obj, color='gray', linestyle='--', alpha=0.5)
        # Place labels only on the first axis to avoid overlap
        if ax == axes[0]:
            ax.text(date_obj, ax.get_ylim()[0], event_name, rotation=90, verticalalignment='bottom', fontsize=10, color='#555')

plt.tight_layout()
plt.show()

### 4.3 OLS regression on Fear and Anger (§5.7)

In [ ]:
import pandas as pd
import statsmodels.api as sm
import numpy as np

# ==========================================
# 1. Feature engineering
# ==========================================
print("Engineering features from full text and metadata...")
df_reg = df_analysis.copy()

# A. Core indicator: is_business
df_reg['is_business'] = (df_reg['functional_frame'] == 'Economic/Rational').astype(int)

# B. Time covariates: date and presidential era
if 'date' not in df_reg.columns:
    df_reg['date'] = pd.to_datetime(df_reg['pub_date'])
df_reg['days_since_start'] = (df_reg['date'] - df_reg['date'].min()).dt.days
# Biden era indicator (>= 2021-01-20)
df_reg['is_biden_era'] = (df_reg['date'] >= '2021-01-20').astype(int)

# C. Genre indicator: is_opinion (check document_type and news_desk)
# 1 if document_type contains 'opinion' or news_desk contains 'Op-Ed'
df_reg['is_opinion'] = df_reg.apply(lambda x: 1 if 'opinion' in str(x['document_type']).lower() 
                                    or 'op-ed' in str(x['news_desk']).lower() else 0, axis=1)

# D. Topic indicators: extracted from full text (the most important step)
# Fillna to avoid errors
texts = df_reg['full_text'].fillna('').astype(str).str.lower()

# Topic keyword lists
keywords_geo = ['taiwan', 'hong kong', 'military', 'navy', 'weapon', 'spy', 'human rights', 'xinjiang']
keywords_covid = ['covid', 'virus', 'pandemic', 'lockdown', 'quarantine', 'zero-covid', 'wuhan']
keywords_tech = ['chip', 'semiconductor', 'technology', 'huawei', 'tiktok', 'artificial intelligence', 'ai']

# Generate 0/1 dummies
df_reg['topic_geopolitics'] = texts.apply(lambda x: 1 if any(k in x for k in keywords_geo) else 0)
df_reg['topic_covid'] = texts.apply(lambda x: 1 if any(k in x for k in keywords_covid) else 0)
df_reg['topic_tech'] = texts.apply(lambda x: 1 if any(k in x for k in keywords_tech) else 0)

# E. Control: article length
df_reg['word_count'] = texts.apply(lambda x: len(x.split()))

# ==========================================
# 2. Run regression
# ==========================================
# Drop rows with missing values
features = [
    'is_business',          
    'topic_geopolitics',    
    'topic_covid',          
    'topic_tech',           
    'is_biden_era',         
    'word_count',           
    'days_since_start'      
]

# Coerce all features to numeric
df_reg = df_reg.dropna(subset=features + ['fear', 'anger'])
for col in features:
    df_reg[col] = pd.to_numeric(df_reg[col])

X = sm.add_constant(df_reg[features])  # add intercept

def print_model_result(y_col_name, title):
    y = df_reg[y_col_name]
    model = sm.OLS(y, X).fit()
    
    print(f"\n{'='*20} {title} {'='*20}")
    print(model.summary())  # full summary including R^2, N, F-stat
    
    coef = model.params['is_business']
    pval = model.pvalues['is_business']
    return coef, pval

# Run Model A: determinants of FEAR
coef_fear, p_fear = print_model_result('fear', "Model A: Determinants of FEAR")

# Run Model B: determinants of ANGER
coef_anger, p_anger = print_model_result('anger', "Model B: Determinants of ANGER")

# ==========================================
# 3. Interpret results
# ==========================================
print("\n=== FINAL INTERPRETATION (with control variables) ===")

print(f"1. [Fear Model] Effect of Business Desk: {coef_fear:.4f} (p={p_fear:.4f})")
if 'topic_covid' in X.columns:
    print(f"   -> Covid topic prevalence: {X['topic_covid'].mean():.2f}")  # for context

print(f"2. [Anger Model] Effect of Business Desk: {coef_anger:.4f} (p={p_anger:.4f})")

if coef_anger < -0.01 and p_anger < 0.05:
    print("\nKEY FINDING: Even after controlling for topics (Taiwan, COVID) and era (Biden),")
    print("   Business articles are still SIGNIFICANTLY LESS ANGRY.")
    print("   This proves the 'Anger Gap' is structural, not just because they cover different topics.")

## 5. Distinctive Keywords: Shanghai Lockdown and Spy Balloon (§5.8)

Term-frequency comparison between Business and Political desks within two narrow event windows. Output is reported in the §5.8 prose rather than as a numbered figure.


### 5.1 Distinctive keywords for Shanghai Lockdown and Spy Balloon windows

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd
import numpy as np

# Stronger stopword list (removes noise observed in prior runs)
my_stops = [
    'mr', 'ms', 'said', 'year', 'years', 'new', 'time', 'state', 'states', 
    'united', 'people', 'like', 'just', 'percent', 'company', 'companies',
    'china', 'chinese', 'beijing', 'government', 'official', 'officials',
    'city', 'week', 'month', 'day', 'world', 'country'
]

# Combine with scikit-learn's native stopwords
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
all_stops = list(ENGLISH_STOP_WORDS) + my_stops

def get_distinctive_keywords(df_window, top_n=10):
    # Split texts by frame
    biz_text = df_window[df_window['functional_frame'] == 'Economic/Rational']['full_text']
    poli_text = df_window[df_window['functional_frame'] == 'Political/Hostile']['full_text']
    
    if len(biz_text) == 0 or len(poli_text) == 0:
        return [], []

    # 1. Vectorize (term counts)
    vec = CountVectorizer(stop_words=all_stops, max_df=0.9, min_df=2)
    try:
        # Fit on combined text to share vocabulary
        X = vec.fit_transform(pd.concat([biz_text, poli_text]))
        feature_names = np.array(vec.get_feature_names_out())
        
        # Compute total term frequency per group
        biz_sum = vec.transform(biz_text).sum(axis=0).A1
        poli_sum = vec.transform(poli_text).sum(axis=0).A1
        
        # 2. Normalize to avoid bias from unequal article counts
        biz_norm = biz_sum / (biz_sum.sum() + 1)
        poli_norm = poli_sum / (poli_sum.sum() + 1)
        
        # 3. Difference score
        # Positive: distinctive to Business; negative: distinctive to Political
        diff_score = biz_norm - poli_norm
        
        # 4. Rank and extract
        # Most distinctive Business terms (highest diff)
        biz_top_indices = diff_score.argsort()[-top_n:][::-1]
        biz_keywords = feature_names[biz_top_indices]
        
        # Most distinctive Political terms (lowest / most negative diff)
        poli_top_indices = diff_score.argsort()[:top_n]
        poli_keywords = feature_names[poli_top_indices]
        
        return biz_keywords, poli_keywords
        
    except ValueError:
        return [], []

# --- Execute ---
print("=== Distinctive keyword analysis ===")
events = {
    "Shanghai Lockdown": ("2022-03-25", "2022-04-25"),
    "Spy Balloon": ("2023-01-25", "2023-02-25")
}

for event_name, (start, end) in events.items():
    print(f"\nEvent: {event_name}")
    mask = (df_analysis['date'] >= start) & (df_analysis['date'] <= end)
    window_df = df_analysis[mask]
    
    biz_keys, poli_keys = get_distinctive_keywords(window_df)
    
    print(f"  Distinctive to BUSINESS desk:")
    print(f"    {list(biz_keys)}")
    
    print(f"  Distinctive to POLITICAL desk:")
    print(f"    {list(poli_keys)}")

## 6. Identity Lexicons and the Crisis–Ideology Tradeoff (§6.1–6.2)

Three theory-driven lexicons (Political Self, Economic Self, Coercive Other) are scored per article and tracked across time and desk type. The China Threat Index and U.S. Free Self Index are then constructed from these scores to test the crisis–ideology negative correlation.


### 6.1 Define identity lexicons

In [ ]:
# ==========================================
# 1. Define identity lexicons
# ==========================================

# A. Political Self: U.S. as "leader of the free world" — democratic / institutional
# Includes values, institutions, and procedural language
USA_POLITICAL_IDENTITY = {
    # Core values
    "democracy", "democratic", "liberty", "freedom", "free speech",
    "rights", "civil liberties", "justice", "equality", "pluralism",
    "individualism", "dignity", "rule of law", "transparency", "accountability",
    
    # Institutions and process (the structural skeleton of U.S. identity)
    "constitution", "constitutional", "checks and balances", "separation of powers",
    "congress", "senate", "house of representatives", "supreme court", "judiciary",
    "bipartisan", "legislation", "voters", "ballot", "election", "electoral",
    "presidency", "white house", "capitol",
    
    # Global role
    "international order", "rules-based", "liberal order", "alliance", "alliances",
    "nato", "partners", "leadership", "diplomacy", "commitment", "credibility"
}

# B. Economic Self: U.S. as "robust market" — capitalist / resilient
# Adds capitalist vitality and recovery vocabulary
USA_ECONOMIC_IDENTITY = {
    # Market dynamics
    "growth", "expansion", "boom", "surge", "climb", "gain",
    "market", "markets", "capitalism", "capitalist", "private sector",
    "enterprise", "entrepreneur", "entrepreneurship", "innovation", "innovative",
    "tech", "technology", "silicon valley", "startups",
    
    # Strength and resilience (the "Strong Self")
    "resilient", "resilience", "robust", "strong", "strength", "solid",
    "recovery", "rebound", "bounce back", "stable", "stability",
    "powerhouse", "dominance", "dominant", "competitive", "competitiveness",
    
    # Outcomes (concrete results)
    "jobs", "employment", "hiring", "wages", "prosperity", "wealth",
    "investment", "spending", "consumer", "demand", "supply chain"
}

# C. The other side of the mirror — what the U.S. is NOT (the "Unfree/Coercive" Other)
# Expanded with concrete control mechanisms and fear-based language
UNFREE_OTHER = {
    # Nature of regime
    "authoritarian", "autocracy", "autocrat", "dictatorship", "dictator",
    "totalitarian", "regime", "police state", "one-party", "communist",
    "strongman", "hardline", "draconian",
    
    # Methods of control
    "censorship", "censor", "surveillance", "spy", "spying", "firewall",
    "crackdown", "purge", "detention", "detain", "jail", "arrest",
    "repression", "repressive", "oppression", "oppressive", "coercion", "coerce",
    "propaganda", "state media", "state-controlled", "indoctrination",
    
    # Human impact
    "human rights abuses", "violation", "forced labor", "concentration camp",
    "dissident", "silence", "restrict", "curb", "violate", "abuse"
}

print("Expanded lexicon defined (higher sensitivity, stronger signal).")

### 6.2 Score articles on identity dimensions

In [ ]:
import re
import pandas as pd
import numpy as np

# ==========================================
# 1. Core utility functions
# ==========================================
def _normalize(text: str) -> str:
    # Force string to avoid NaN errors
    if not isinstance(text, str): return str(text) if text is not None else ""
    text = text.lower()
    text = re.sub(r"\s+", " ", text)
    return text

def _count_lexicon(text: str, lexicon: set) -> int:
    t = _normalize(text)
    total = 0
    for term in lexicon:
        term_l = term.lower()
        if " " in term_l:  # phrase match
            total += t.count(term_l)
        else:              # word-boundary match
            total += len(re.findall(rf"\b{re.escape(term_l)}\b", t))
    return total

# ==========================================
# 2. Main scoring function
# ==========================================
def add_identity_scores(df: pd.DataFrame, text_col="full_text") -> pd.DataFrame:
    """
    Compute per-article scores for Political Identity, Economic Identity, and Unfree Score.
    """
    out = df.copy()
    
    # Check column existence
    if text_col not in out.columns:
        print(f"ERROR: column '{text_col}' not found in DataFrame")
        print(f"   Available columns: {out.columns.tolist()}")
        return out

    print(f"Computing term frequencies on '{text_col}' (this may take a moment)...")
    
    # Compute total word count (for normalization)
    out["word_count"] = out[text_col].apply(lambda x: max(1, len(_normalize(str(x)).split())))
    scale = out["word_count"] / 1000.0  # normalize: counts per 1000 words

    # Compute raw scores for the three dimensions
    out["score_political_raw"] = out[text_col].apply(lambda x: _count_lexicon(str(x), USA_POLITICAL_IDENTITY))
    out["score_economic_raw"] = out[text_col].apply(lambda x: _count_lexicon(str(x), USA_ECONOMIC_IDENTITY))
    out["score_unfree_raw"] = out[text_col].apply(lambda x: _count_lexicon(str(x), UNFREE_OTHER))

    # Compute normalized scores (density)
    out["US_Political_Identity"] = out["score_political_raw"] / scale
    out["US_Economic_Strength"] = out["score_economic_raw"] / scale
    out["Unfree_Otherness"] = out["score_unfree_raw"] / scale

    # Binary opposition score (Freedom Score)
    out["Freedom_Index"] = out["US_Political_Identity"] - out["Unfree_Otherness"]

    return out

# ==========================================
# 3. Execute scoring
# ==========================================
# Ensure df_text is in scope
if 'df_text' in locals():
    print("Processing df_text...")
    # Use the 'full_text' column
    df_scored = add_identity_scores(df_text, text_col="full_text")
    
    print("\nScoring complete. First 5 rows:")
    # Show headline if available; otherwise just the scores
    show_cols = ['US_Political_Identity', 'Freedom_Index']
    if 'headline' in df_scored.columns:
        show_cols.insert(0, 'headline')
    
    print(df_scored[show_cols].head())
else:
    print("ERROR: 'df_text' not in memory. Run the data-loading cell first.")

### 6.3 Figure 12 — Identity register evolution across desks (§6.1)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import pandas as pd

# ==========================================
# 3. Visualization: The Tale of Two Americas (Thesis-Safe Version)
# ==========================================

def plot_identity_evolution(df_scored, window=3):
    # 1. Prepare time data
    d = df_scored.copy()
    d["pub_date"] = pd.to_datetime(d["pub_date"])
    d["month"] = d["pub_date"].dt.to_period("M").dt.to_timestamp()
    
    # Auto-detect time range (avoid hard-coded years in title)
    start_year = d["pub_date"].min().year
    end_year = d["pub_date"].max().year
    
    # 2. Define desk groupings
    political_desks = ['Foreign', 'Washington', 'Politics', 'World', 'National']
    business_desks = ['Business', 'SundayBusiness', 'Business Day']
    
    target_desks = political_desks + business_desks
    d_filtered = d[d['news_desk'].isin(target_desks)].copy()
    
    # 3. Mapping logic
    def map_desk(desk_name):
        return 'Business Desk' if desk_name in business_desks else 'Political/Foreign Desk'

    d_filtered['Desk_Type'] = d_filtered['news_desk'].apply(map_desk)

    # 4. Metrics
    metrics = [
        ("US_Political_Identity", "The 'Democratic' Self (Political Values)"),
        ("US_Economic_Strength", "The 'Market' Self (Economic Resilience)"),
        ("Unfree_Otherness", "The Construction of 'Otherness' (Autocracy/Threat)")
    ]

    # 5. Plot
    fig, axes = plt.subplots(3, 1, figsize=(12, 14), sharex=True)
    colors = {'Political/Foreign Desk': '#d62728', 'Business Desk': '#1f77b4'}

    for i, (metric, title) in enumerate(metrics):
        ax = axes[i]
        
        for desk, g in d_filtered.groupby("Desk_Type"):
            # Rolling mean
            ts = g.groupby("month")[metric].mean().rolling(window=window, min_periods=1).mean()
            ax.plot(ts.index, ts.values, label=desk, color=colors[desk], linewidth=3, alpha=0.9)
        
        ax.set_title(title, fontsize=14, fontweight='bold', pad=10)
        ax.set_ylabel("Freq per 1k words", fontsize=11)
        ax.grid(True, linestyle='--', alpha=0.5)
        
        if i == 0:
            ax.legend(loc="upper left", title="Editorial Lens", fontsize=11, frameon=True)

    # 6. X-axis formatting
    plt.xlabel("Date", fontsize=12)
    axes[-1].xaxis.set_major_locator(mdates.MonthLocator(interval=6))
    axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
    plt.xticks(rotation=45)
    
    # Use academic phrasing ("Bifurcated") and auto-filled years in title
    safe_title = f"Bifurcated Identity: Divergent Framing in NYT Desks ({start_year}-{end_year})"
    plt.suptitle(safe_title, fontsize=16, y=0.92, fontweight='bold')
    plt.subplots_adjust(top=0.88)
    
    plt.show()

# Generate plot
print("Generating visualization (academic version)...")
plot_identity_evolution(df_scored)

### 6.4 Construct China Threat Index and U.S. Free Self Index

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
import numpy as np

# ==========================================
# 1. Prepare data (using df_text)
# ==========================================
print("Preparing data...")

# Ensure df_text is in scope
if 'df_text' in locals():
    # Work on a copy to avoid mutating df_text
    df_analysis = df_text.copy()
    
    # Text column
    text_col = 'full_text'
    
    # Check column existence
    if text_col not in df_analysis.columns:
        print(f"ERROR: column '{text_col}' not found in df_text")
        # Print available columns for debugging
        print(f"   Available columns: {df_analysis.columns.tolist()}")
    else:
        # 1. Parse pub_date as datetime
        if 'pub_date' in df_analysis.columns:
            df_analysis['pub_date'] = pd.to_datetime(df_analysis['pub_date'], errors='coerce')
            # Drop rows with unparseable dates
            df_analysis = df_analysis.dropna(subset=['pub_date'])
        else:
            print("ERROR: df_text is missing the 'pub_date' column.")

        # 2. Fill empty text to avoid errors
        df_analysis[text_col] = df_analysis[text_col].fillna("")

        # ==========================================
        # 2. Define lexicons
        # ==========================================
        
        # A. Trigger: China threat
        LEX_CHINA_THREAT = {
            # Specific crisis events
            "balloon", "spy", "surveillance", "espionage",
            "taiwan", "strait", "invasion", "military drill",
            "tiktok", "data", "app", "ban",
            # General threat language
            "threat", "aggression", "coerce", "coercion", "intimidate",
            "undermine", "challenge", "conflict", "danger", "risk",
            "weapon", "war", "battle", "confrontation"
        }

        # B. Response: U.S. Free Self
        LEX_FREE_SELF = {
            "democracy", "democratic", "liberty", "freedom", "free speech",
            "rights", "civil liberties", "rule of law", "constitution",
            "checks and balances", "transparency", "pluralism",
            "international order", "rules-based", "alliances", "leadership",
            "values", "human rights", "justice"
        }

        # ==========================================
        # 3. Compute index intensity
        # ==========================================
        def count_lexicon(text, lexicon):
            if not isinstance(text, str): return 0
            text = text.lower()
            count = 0
            for term in lexicon:
                if " " in term:  # phrase
                    count += text.count(term)
                else:  # single word
                    count += len(re.findall(rf"\b{re.escape(term)}\b", text))
            return count

        print("Computing Threat vs. Freedom indices on full text...")

        # Compute total word count (for normalization)
        # full_text is long; use simple split for word count
        df_analysis['word_count'] = df_analysis[text_col].apply(lambda x: len(str(x).split()))
        
        # Drop articles with fewer than 50 words
        df_analysis = df_analysis[df_analysis['word_count'] > 50].copy()

        # Compute raw frequencies
        df_analysis['score_threat'] = df_analysis[text_col].apply(lambda x: count_lexicon(x, LEX_CHINA_THREAT))
        df_analysis['score_free'] = df_analysis[text_col].apply(lambda x: count_lexicon(x, LEX_FREE_SELF))

        # Normalize: keywords per 1,000 words
        df_analysis['China_Threat_Index'] = (df_analysis['score_threat'] / df_analysis['word_count']) * 1000
        df_analysis['US_Free_Self_Index'] = (df_analysis['score_free'] / df_analysis['word_count']) * 1000

        # ==========================================
        # 4. Time series and plotting
        # ==========================================
        # Monthly aggregation
        df_analysis['month'] = df_analysis['pub_date'].dt.to_period('M').dt.to_timestamp()
        monthly_trends = df_analysis.groupby('month')[['China_Threat_Index', 'US_Free_Self_Index']].mean()
        
        # 3-month rolling mean
        monthly_smooth = monthly_trends.rolling(window=3).mean().dropna()

        # Compute correlation
        correlation = monthly_smooth['China_Threat_Index'].corr(monthly_smooth['US_Free_Self_Index'])



### 6.5 Figure 13 — Crisis–Ideology see-saw plot (§6.2)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

# 2. Plot
fig, ax1 = plt.subplots(figsize=(14, 7))

# Set background grid
ax1.grid(True, linestyle='--', alpha=0.3)

# Threat line (solid red, with filled area)
color_threat = '#d62728'  # red
ax1.set_xlabel('Date', fontsize=12)
ax1.set_ylabel('China Threat (Crisis Mode)', color=color_threat, fontsize=14, fontweight='bold')
ax1.plot(monthly_smooth.index, monthly_smooth['China_Threat_Index'], 
         color=color_threat, linewidth=3, label='External Threat (Trigger)')
# Fill red area to emphasize crisis periods
ax1.fill_between(monthly_smooth.index, monthly_smooth['China_Threat_Index'], 
                 color=color_threat, alpha=0.1)
ax1.tick_params(axis='y', labelcolor=color_threat)

# Free Self line (dashed blue, twin axis)
# Keep right axis upright (inverted axes obscure rather than reveal the divergence)
ax2 = ax1.twinx()
color_free = '#1f77b4'  # blue
ax2.set_ylabel('US Free Self (Ideological Mode)', color=color_free, fontsize=14, fontweight='bold')
ax2.plot(monthly_smooth.index, monthly_smooth['US_Free_Self_Index'], 
         color=color_free, linewidth=3, linestyle='--', label='Internal Identity (Response)')
ax2.tick_params(axis='y', labelcolor=color_free)

# 3. Annotate divergence moments
# Identify representative divergence points
# E.g., the 2023-02 balloon incident: high threat, low ideology
events = {
    '2022-08-01': 'Pelosi Visit\n(High Threat, Low Ideology)',
    '2021-03-18': 'Alaska Summit\n(High Ideology, Low Threat)' 
    # Alaska Summit: ideological confrontation without active conflict
}

for date_str, text in events.items():
    date_obj = pd.to_datetime(date_str)
    if date_obj >= monthly_smooth.index.min() and date_obj <= monthly_smooth.index.max():
        ax1.axvline(x=date_obj, color='gray', linestyle=':', alpha=0.8)
        # Annotate
        ax1.text(date_obj, ax1.get_ylim()[1]*0.8, text, 
                 rotation=0, color='black', fontsize=10, 
                 bbox=dict(facecolor='white', alpha=0.8, edgecolor='gray'))

plt.title('The "Crisis-Ideology" See-Saw: Negative Correlation (-0.42)\n(Operational Security crowds out Abstract Values)', 
          fontsize=16, fontweight='bold', pad=20)

fig.tight_layout()
plt.show()

## 7. Word Embedding Analysis (§7)

A skip-gram Word2Vec model (300-dim, window 10, min_count 5, 30 epochs, 10-iteration bootstrap) is trained on the full-text China-related corpus following the semantic-geometry methodology of Kozlowski, Taddy, and Evans (2019). Year-by-year models are also trained for the longitudinal analysis. The structural-break analysis on annual shift magnitudes identifies 2022 as the rupture year.


### 7.1 Preprocessing: cleaning, stopwords, bigram phrase detection

In [ ]:
import gensim
from gensim.utils import simple_preprocess
from gensim.parsing.preprocessing import remove_stopwords, STOPWORDS
from gensim.models.phrases import Phrases, Phraser
import pandas as pd

# ==========================================
# 1. NYT-specific stopword list
# ==========================================

# 1. Generic filler words (note: 'new' kept out so 'new_york' can survive)
base_stops = {
    'many', 'much', 'more', 'most', 'other', 'same', 'such', 'own', 'first', 'last', 
    'top', 'major', 'big', 'large', 'good', 'bad', 'great', 'high', 'low', 'second', 'third',
    'hard', 'early', 'late', 'likely', 'possible', 'able', 'full', 'small', 'recent', 'long', 
    'little', 'huge', 'vast', 'short', 'broad', 'wide', 'single', 'several', 'various', 
    'main', 'general', 'real', 'whole', 'entire', 'important', 'significant', 'key',
    'said', 'mr', 'ms', 'mrs', 'dr', 'prof', 'advertisement', 'supported', 'reporting' 
}

# 2. Administrative / directional
geo_stops = {
    'central', 'local', 'federal', 'national', 'global', 'international', 'domestic',
    'foreign', 'western', 'eastern', 'southern', 'northern', 'southeastern', 
    'municipal', 'regional', 'urban', 'rural', 'overseas', 'internal', 'external'
}

# 3. Temporal
time_stops = {
    'former', 'senior', 'current', 'past', 'next', 'previous', 'future', 
    'expected', 'potential', 'years', 'days', 'weeks', 'months'
}

# 4. Merge final stopword list (Gensim defaults + custom)
final_stop_words = STOPWORDS.union(base_stops).union(geo_stops).union(time_stops)

# ------------------------------------------
# Cleaning function
# ------------------------------------------
def clean_text(text):
    # 1. Remove standard stopwords with Gensim (the, is, at, ...)
    text = remove_stopwords(text)
    # 2. Tokenize (lowercase, strip punctuation)
    tokens = simple_preprocess(text, deacc=True)
    # 3. Remove custom high-level stopwords
    return [token for token in tokens if token not in final_stop_words]

# ==========================================
# 2. Execute cleaning and phrase detection
# ==========================================

print("1/2: tokenizing and cleaning...")
# df_text must be the corpus DataFrame
basic_sentences = df_text['full_text'].apply(clean_text).tolist()

print("2/2: training phrase model (e.g., united_states, xi_jinping)...")
# Train bigram model
# min_count=5: minimum phrase frequency
# threshold=10: higher = more conservative phrase merging
bigram_model = Phrases(basic_sentences, min_count=5, threshold=10)
bigram_phraser = Phraser(bigram_model)

# Apply phrase model to all sentences
final_sentences = [bigram_phraser[sent] for sent in basic_sentences]

print("Data preparation complete.")
print(f"Original tokens example: {basic_sentences[0][:10]}")
print(f"Phrased tokens example: {final_sentences[0][:10]}")
# Check output for bigrams like 'united_states' or 'human_rights'

### 7.2 Figure 14 — Word2Vec training (Paper Mode) and static projection (§7.1)

In [ ]:
import multiprocessing
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from gensim.models import Word2Vec
from time import time
from tqdm import tqdm

# ==========================================
# 1. Configuration: paper-grade precision (Paper Mode)
# ==========================================
# Fast Mode was for quick validation; this is the production run
N_ITER = 10       # 10 iterations averaged (academic standard)
N_EPOCHS = 30     # 30 epochs (deep convergence)

# Target term list
TARGET_GROUPS = {
    "🇨🇳 PRC State":    ['china', 'beijing', 'chinese_government', 'communist_party', 'ccp', 'xi_jinping'],
    "🔥 Hotspots":     ['hong_kong', 'xinjiang', 'taiwan', 'taipei'],
    "🇺🇸 US Identity":  ['united_states', 'america', 'washington', 'biden', 'trump'],
    "🇯🇵 Allies":       ['japan']
}
ALL_TARGETS = [t for group in TARGET_GROUPS.values() for t in group]

# ==========================================
# 2. Execute high-precision bootstrap
# ==========================================
def run_paper_mode(input_data):
    cores = multiprocessing.cpu_count()
    workers = max(1, cores - 1)
    
    print("Starting final training (Paper Mode)...")
    print(f"   Config: Iterations={N_ITER} | Epochs={N_EPOCHS} | Cores={workers}")
    print("   Estimated runtime: 15-20 minutes...")
    
    # Storage for per-iteration results
    raw_data = {t: {'pol': [], 'eco': []} for t in ALL_TARGETS}
    
    start_time = time()
    
    for i in tqdm(range(N_ITER), desc="Training Progress"):
        # Train
        model = Word2Vec(input_data, vector_size=300, window=10, min_count=5, 
                         workers=workers, sg=1, epochs=N_EPOCHS, seed=i)
        
        # Define axes (bigram-aware seed terms)
        pol_pos = ['democracy', 'freedom', 'liberty', 'human_rights', 'democratic']
        pol_neg = ['autocracy', 'dictatorship', 'repression', 'authoritarian', 'communist_party']
        eco_pos = ['market', 'trade', 'growth', 'investment', 'business', 'opportunity']
        eco_neg = ['threat', 'risk', 'conflict', 'national_security', 'espionage']
        
        # Compute axis vectors
        def get_axis_vec(pos, neg):
            p = [model.wv[w] for w in pos if w in model.wv]
            n = [model.wv[w] for w in neg if w in model.wv]
            if not p or not n: return None
            return np.mean(p, axis=0) - np.mean(n, axis=0)
            
        pol_axis = get_axis_vec(pol_pos, pol_neg)
        eco_axis = get_axis_vec(eco_pos, eco_neg)
        
        # Projection
        for target in ALL_TARGETS:
            if target in model.wv and pol_axis is not None:
                # Political-axis projection
                raw_data[target]['pol'].append(np.dot(model.wv[target], pol_axis) / np.linalg.norm(pol_axis))
                # Economic-axis projection
                raw_data[target]['eco'].append(np.dot(model.wv[target], eco_axis) / np.linalg.norm(eco_axis))

    total_time = (time() - start_time) / 60
    print(f"\nTraining complete. Total runtime: {total_time:.1f} minutes")
    
    # Aggregate results
    final_df = []
    for group, terms in TARGET_GROUPS.items():
        for term in terms:
            if raw_data[term]['pol']:
                pol_mean = np.mean(raw_data[term]['pol'])
                eco_mean = np.mean(raw_data[term]['eco'])
                # Compute standard deviation (for error bars in the manuscript)
                pol_std = np.std(raw_data[term]['pol'])
                eco_std = np.std(raw_data[term]['eco'])
                
                final_df.append({
                    'Group': group,
                    'Term': term,
                    'Pol_Score': pol_mean,
                    'Eco_Score': eco_mean,
                    'Gap': eco_mean - pol_mean,
                    'Pol_Std': pol_std,  # kept for downstream use
                    'Eco_Std': eco_std
                })
    
    return pd.DataFrame(final_df)

# ==========================================
# 3. Run and save
# ==========================================
# Uses the cleaned final_sentences
df_results = run_paper_mode(final_sentences)

# Save table to CSV
df_results.to_csv("NYT_Identity_Final_Results.csv", index=False)
print("\nResults saved to NYT_Identity_Final_Results.csv")
display(df_results)  # rendered as table in Jupyter

# ==========================================
# 4. Final publication-ready plot
# ==========================================
plt.figure(figsize=(12, 10), dpi=300)  # 300 DPI for print

# Background shading
plt.axvspan(0, 1.5, ymin=0.5, ymax=1, color='green', alpha=0.03)  # democratic-partner zone
plt.axvspan(-1.5, 0, ymin=0.5, ymax=1, color='red', alpha=0.03)   # decoupling zone

# Axes
plt.axhline(0, color='black', linewidth=1, alpha=0.5)
plt.axvline(0, color='black', linewidth=1, alpha=0.5)

# Style mapping
styles = {
    "PRC State":      {'c': '#d62728', 'm': 'o'},   # red circles
    "Hotspots":       {'c': '#ff7f0e', 'm': 's'},   # orange squares
    "US Identity":    {'c': '#1f77b4', 'm': '^'},   # blue triangles
    "Allies":         {'c': '#2ca02c', 'm': 'D'}    # green diamonds
}

# Plot points
for idx, row in df_results.iterrows():
    style = styles.get(row['Group'], {'c': 'grey', 'm': 'x'})
    
    # Enlarge markers for CCP and China
    size = 300 if row['Term'] in ['china', 'communist_party', 'united_states'] else 150
    marker = '*' if row['Term'] == 'communist_party' else style['m']
    
    plt.scatter(row['Pol_Score'], row['Eco_Score'], 
                c=style['c'], s=size, marker=marker, 
                edgecolors='white', linewidth=1.5, alpha=0.9, label=row['Group'])
    
    # Labels
    plt.text(row['Pol_Score']+0.02, row['Eco_Score']+0.02, row['Term'], 
             fontsize=12, fontweight='bold', alpha=0.8)

# Deduplicate legend entries
handles, labels = plt.gca().get_legend_handles_labels()
by_label = dict(zip(labels, handles))
plt.legend(by_label.values(), by_label.keys(), loc='lower left')

# Title and axis labels
plt.title('The Geometry of Geopolitics: Final Projections (NYT 2020-2024)', fontsize=16, pad=20)
plt.xlabel('Political Dimension (Autocracy <---> Democracy)', fontsize=12)
plt.ylabel('Economic Dimension (Threat <---> Opportunity)', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.3)
plt.xlim(-1.4, 0.6)
plt.ylim(-0.6, 0.6)

# Save figure
plt.savefig("Final_Geometry_Plot.png", bbox_inches='tight')
print("Figure saved to Final_Geometry_Plot.png")

plt.show()

### 7.3 Figures 15, 16 — Year-by-year Word2Vec models and entity trajectories (§7.2)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from gensim.models import Word2Vec

# ==========================================
# 1. Config: expanded term list (more U.S.-related terms)
# ==========================================
TARGET_GROUPS = {
    "PRC State":   ['china', 'ccp', 'xi_jinping', 'beijing'],
    "Hotspots":    ['taiwan', 'hong_kong', 'xinjiang'],
    "US Identity": ['united_states', 'biden', 'trump', 'washington']  # added washington
}

# Axis definitions (unchanged)
AXIS_DEFINITIONS = {
    "pol_pos": ['democracy', 'freedom', 'human_rights'],
    "pol_neg": ['autocracy', 'dictatorship', 'authoritarian'],
    "eco_pos": ['market', 'trade', 'growth', 'business'],
    "eco_neg": ['threat', 'risk', 'security', 'conflict']
}

# ==========================================
# 2. Core functions (robust mode)
# ==========================================
def get_projection_score(model, word, pos_seeds, neg_seeds):
    if word not in model.wv:
        return np.nan
    p_vecs = [model.wv[w] for w in pos_seeds if w in model.wv]
    n_vecs = [model.wv[w] for w in neg_seeds if w in model.wv]
    if not p_vecs or not n_vecs:
        return np.nan
    axis_vec = np.mean(p_vecs, axis=0) - np.mean(n_vecs, axis=0)
    score = np.dot(model.wv[word], axis_vec) / np.linalg.norm(axis_vec)
    return score

def run_yearly_evolution_robust(df):
    df = df.copy()
    
    # Auto-detect date column
    possible_date_cols = [c for c in df.columns if 'date' in str(c).lower()]
    if not possible_date_cols:
        print("ERROR: no date column found.")
        return pd.DataFrame()
    
    target_date_col = possible_date_cols[0]
    df[target_date_col] = pd.to_datetime(df[target_date_col])
    df['year'] = df[target_date_col].dt.year
    years = sorted(df['year'].unique())
    
    print(f"Analyzed years: {years}")
    
    yearly_data = []
    
    for year in years:
        sentences = df[df['year'] == year]['final_sentences'].tolist()
        
        # Guard against very low data
        if len(sentences) < 5: 
            continue
            
        # Train with lower min_count so target terms are not filtered out
        # min_count=2: keep terms occurring at least twice in the year
        model = Word2Vec(sentences, vector_size=300, window=10, min_count=2, 
                         workers=4, seed=42, epochs=40) 
        
        for group_name, terms in TARGET_GROUPS.items():
            for term in terms:
                pol = get_projection_score(model, term, AXIS_DEFINITIONS['pol_pos'], AXIS_DEFINITIONS['pol_neg'])
                eco = get_projection_score(model, term, AXIS_DEFINITIONS['eco_pos'], AXIS_DEFINITIONS['eco_neg'])
                
                if not np.isnan(pol) and not np.isnan(eco):
                    yearly_data.append({
                        'Year': year, 
                        'Group': group_name, 
                        'Term': term, 
                        'Pol': pol, 
                        'Eco': eco
                    })
    
    return pd.DataFrame(yearly_data)

# ==========================================
# 3. Run analysis
# ==========================================
print("Recomputing with expanded term list...")
df_res = run_yearly_evolution_robust(df_text)

if df_res.empty:
    print("WARNING: empty results.")
else:
    print(f"Computation complete. {len(df_res)} data points.")
    
    # Compute shift magnitudes
    df_res = df_res.sort_values(['Term', 'Year'])
    df_res['Prev_Pol'] = df_res.groupby('Term')['Pol'].shift(1)
    df_res['Prev_Eco'] = df_res.groupby('Term')['Eco'].shift(1)
    df_res['Magnitude'] = np.sqrt((df_res['Pol']-df_res['Prev_Pol'])**2 + (df_res['Eco']-df_res['Prev_Eco'])**2)
    
    # Save results
    df_res.to_csv("Yearly_Changes_Expanded.csv", index=False)

    # ==========================================
    # 4. Visualization: trajectory plot (including U.S. terms)
    # ==========================================
    plt.figure(figsize=(12, 12))
    
    # Terms to plot (now includes U.S. identity terms)
    plot_terms = ['china', 'ccp', 'taiwan', 'united_states', 'biden', 'trump']
    
    # Color mapping
    colors = {
        'china': 'red', 'ccp': '#8B0000', 'taiwan': 'orange',         # China group
        'united_states': 'blue', 'biden': '#1E90FF', 'trump': '#00008B'  # U.S. group
    }
    
    for term in plot_terms:
        subset = df_res[df_res['Term'] == term]
        if len(subset) < 2: continue 
        
        # Get color (default grey)
        color = colors.get(term, 'grey')
        
        # Plot trajectory
        plt.plot(subset['Pol'], subset['Eco'], marker='o', markersize=6, 
                 label=term, color=color, linewidth=2, alpha=0.7)
        
        # Annotate first and last year only to avoid clutter
        # First year
        first = subset.iloc[0]
        plt.text(first['Pol'], first['Eco']+0.02, str(int(first['Year'])), 
                 fontsize=8, color=color, fontweight='bold')
        # Last year
        last = subset.iloc[-1]
        plt.text(last['Pol'], last['Eco']+0.02, str(int(last['Year'])), 
                 fontsize=10, color=color, fontweight='bold')

        # Direction arrow on the final segment
        if len(subset) >= 2:
            second_last = subset.iloc[-2]
            plt.arrow(second_last['Pol'], second_last['Eco'], 
                      last['Pol']-second_last['Pol'], last['Eco']-second_last['Eco'],
                      head_width=0.02, color=color, length_includes_head=True)

    # Background
    plt.axhline(0, color='black', alpha=0.3)
    plt.axvline(0, color='black', alpha=0.3)
    
    # Quadrant labels
    plt.text(0.4, 0.4, "Democratic Opportunity\n(Allies/Markets)", fontsize=10, color='green', alpha=0.5, ha='center')
    plt.text(-0.4, -0.4, "Autocratic Threat\n(Rivals/Risks)", fontsize=10, color='red', alpha=0.5, ha='center')

    plt.title('US-China Narrative Trajectories (2020-2024)', fontsize=16, pad=20)
    plt.xlabel('Political (Autocracy <--> Democracy)', fontsize=12)
    plt.ylabel('Economic (Threat <--> Opportunity)', fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.3)
    plt.legend(loc='upper right')
    
    plt.show()
    
    # ==========================================
    # 5. Visualization: shift magnitudes by year
    # ==========================================
    plt.figure(figsize=(14, 6))
    
    # Show shift magnitudes for selected core terms
    bar_terms = ['china', 'taiwan', 'united_states']
    bar_data = df_res[df_res['Term'].isin(bar_terms)]
    
    sns.barplot(data=bar_data, x='Year', y='Magnitude', hue='Term', palette='Set1')
    
    plt.title('Yearly Volatility: Which Concept Shifted Most?', fontsize=14, fontweight='bold')
    plt.ylabel('Shift Distance', fontsize=12)
    plt.grid(axis='y', alpha=0.3)
    plt.legend(title='Concept')
    plt.tight_layout()
    plt.show()

### 7.4 Figure 17 (panel a) — Annual shift magnitudes and cut-year identification (§7.3)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ==========================================
# 1. Reload prior results (in case variables were lost)
# ==========================================
try:
    # Read the CSV produced upstream
    df_res = pd.read_csv("Yearly_Changes_Expanded.csv")
    print("Loaded Yearly_Changes_Expanded.csv")
except:
    print("WARNING: CSV not found. Run the previous block to generate it.")
    # If absent, the next steps will fail. Run the previous block first.

# ==========================================
# 2. "Cut year" identification
# ==========================================
# Logic:
shock_by_year = (df_res.dropna(subset=['Magnitude'])
                 .groupby('Year')['Magnitude']
                 .agg(mean_shock='mean', median_shock='median', n='count')
                 .reset_index())

# Find the year with the largest mean and median shocks
cut_mean   = shock_by_year.loc[shock_by_year['mean_shock'].idxmax(), 'Year']
cut_median = shock_by_year.loc[shock_by_year['median_shock'].idxmax(), 'Year']

# ==========================================
# 3. Output and interpretation
# ==========================================
print("\nYearly shock statistics:")
display(shock_by_year.style.background_gradient(cmap='Reds', subset=['mean_shock', 'median_shock']))

print("\nCut-year candidates:")
print(f"   Cut (mean)   : {cut_mean}  (year with largest average shift across terms)")
print(f"   Cut (median) : {cut_median} (year with largest median shift across terms)")

# ==========================================
# 4. Visual verification
# ==========================================
plt.figure(figsize=(10, 5))
sns.barplot(data=shock_by_year, x='Year', y='mean_shock', color='firebrick', alpha=0.7)
plt.axvline(x=shock_by_year[shock_by_year['Year']==cut_mean].index[0], color='black', linestyle='--', label=f'Cut Year: {cut_mean}')
plt.title(f'Systemic Shock: The "Cut Year" is {cut_mean}', fontsize=14, fontweight='bold')
plt.ylabel('Average Magnitude of Shift', fontsize=12)
plt.legend()
plt.show()

### 7.5 Figure 17 (panel b) — SSE-based structural-break test (§7.3)

In [ ]:
years = sorted(shock_by_year['Year'].unique())

def sse_split(cut):
    pre  = shock_by_year[shock_by_year['Year'] <= cut]['mean_shock'].values
    post = shock_by_year[shock_by_year['Year'] >  cut]['mean_shock'].values
    if len(pre) < 2 or len(post) < 2:
        return np.inf
    return ((pre - pre.mean())**2).sum() + ((post - post.mean())**2).sum()

candidates = years[1:-1]  # avoid trivial splits
best_cut = min(candidates, key=sse_split)
print("Best 2-regime cut (SSE):", best_cut)

## 8. Pre/Post-2022 Nearest-Neighbor Analysis and Qualitative Retrieval (§7.4)

Compare semantic neighbors of key entities before and after the 2022 rupture. Qualitative retrieval functions extract the highest-relevance articles per month for the 2022 anatomy reported in §7.4.


### 8.1 Compare neighbors: 'china' and 'taiwan' in 2021 vs. 2022

In [ ]:
import re
import pandas as pd
from gensim.models import Word2Vec

# ==========================================
# 1. Stopword blacklist
# ==========================================
# These tokens are removed from the neighbor results
CUSTOM_STOPWORDS = {
    # Generic filler
    'the', 'a', 'an', 'in', 'on', 'at', 'of', 'for', 'to', 'and', 'is', 'was', 'are', 'were',
    'its', 'itself', 'it', 'that', 'this', 'but', 'by', 'from', 'as', 'with', 'about',
    'country', 'countries', 'nation', 'nations', 'state', 'states',
    'relationship', 'relations', 'part', 'side', 'world', 'year', 'years', 'time',
    'beijing', 'beijings', 'china', 'chinas', 'chinese',  # filler when querying 'china'
    'taiwan', 'taiwans', 'taiwanese', 'taipei', 'island', 'islands'  # filler when querying 'taiwan'
}

# ==========================================
# 2. Strong filter
# ==========================================
def get_clean_neighbors(model, target_word, top_n=10):
    # Preprocess target word
    target_clean = re.sub(r'[^\w]', '', target_word.lower())
    
    if target_word not in model.wv:
        return ["Not Found"] * top_n, [0.0] * top_n

    # Pull 100 candidates so we can drop aggressively
    candidates = model.wv.most_similar(target_word, topn=100)
    
    final_words = []
    final_scores = []
    seen_roots = set()
    
    for word, score in candidates:
        # 1. Strip non-alphanumerics ("China's" -> "chinas", "“strategic" -> "strategic")
        word_clean = re.sub(r'[^\w]', '', word.lower())
        
        # --- Filter checkpoints ---
        
        # Check 1: empty
        if not word_clean: continue
        
        # Check 2: blacklist (core step)
        if word_clean in CUSTOM_STOPWORDS: continue
            
        # Check 3: target word itself
        if target_clean in word_clean: continue  # e.g. when searching 'china', drop 'pro-china'

        # Check 4: root deduplication (e.g. drop 'militaries' if 'military' kept)
        # Use first 5 letters as a rough fingerprint
        root = word_clean[:5] if len(word_clean) > 5 else word_clean
        if root in seen_roots: continue
            
        # --- Passed all filters ---
        seen_roots.add(root)
        final_words.append(word_clean)  # cleaned form
        final_scores.append(score)
        
        if len(final_words) >= top_n:
            break
            
    return final_words, final_scores

# ==========================================
# 3. Comparison function
# ==========================================
def compare_neighbors_ultra(df, year1, year2, target_word='china', top_n=10):
    print(f"Cleaned neighbor comparison for '{target_word}' ({year1} vs {year2})")
    
    # Train
    s1 = df[df['year'] == year1]['final_sentences'].tolist()
    s2 = df[df['year'] == year2]['final_sentences'].tolist()
    
    if len(s1) < 5 or len(s2) < 5: return

    m1 = Word2Vec(s1, vector_size=300, window=10, min_count=5, seed=42, epochs=20)
    m2 = Word2Vec(s2, vector_size=300, window=10, min_count=5, seed=42, epochs=20)
    
    # Fetch neighbors
    w1, s1 = get_clean_neighbors(m1, target_word, top_n)
    w2, s2 = get_clean_neighbors(m2, target_word, top_n)
    
    # Display
    res = pd.DataFrame({
        f'{year1} (Pre)': w1,
        'Sim 1': [f"{s:.2f}" for s in s1],
        '   |   ': ['|'] * top_n,
        f'{year2} (Post)': w2,
        'Sim 2': [f"{s:.2f}" for s in s2]
    })
    
    display(res)

# ==========================================
# 4. Run comparisons
# ==========================================
compare_neighbors_ultra(df_text, 2021, 2022, 'china')
compare_neighbors_ultra(df_text, 2021, 2022, 'taiwan')

### 8.2 Compare neighbors: 'america', 'democracy', 'freedom' in 2021 vs. 2022

In [ ]:
import re
import pandas as pd
from gensim.models import Word2Vec

# ==========================================
# 1. Refined stopword list
# ==========================================
CUSTOM_STOPWORDS = {
    # Generic filler
    'the', 'a', 'an', 'in', 'on', 'at', 'of', 'for', 'to', 'and', 'is', 'was', 'are', 'were',
    'its', 'itself', 'it', 'that', 'this', 'but', 'by', 'from', 'as', 'with', 'about',
    'country', 'countries', 'nation', 'nations', 'state', 'states',
    'relationship', 'relations', 'part', 'side', 'world', 'year', 'years', 'time',
    'government', 'official', 'officials', 'administration', 
    
    # Stopwords for the U.S. self-analysis:
    # We query 'america', so block its variants to avoid trivial synonyms
    'america', 'american', 'americans', 'usa', 'us', 'united', 'washington',
    'biden', 'trump', 'president', 'white_house'
}

# ==========================================
# 2. Core helper (unchanged)
# ==========================================
def get_clean_neighbors(model, target_word, top_n=10):
    # Preprocess target word
    target_clean = re.sub(r'[^\w]', '', target_word.lower())
    
    # If the term is not in the model (e.g. typo), return Not Found
    if target_word not in model.wv:
        return ["Not Found"] * top_n, [0.0] * top_n

    # Pull 100 candidates
    candidates = model.wv.most_similar(target_word, topn=100)
    
    final_words = []
    final_scores = []
    seen_roots = set()
    
    # Add the target word and its plural to seen_roots
    seen_roots.add(target_clean)
    seen_roots.add(target_clean + 's') 
    
    for word, score in candidates:
        word_clean = re.sub(r'[^\w]', '', word.lower())
        
        # --- Filter checkpoints ---
        if not word_clean: continue
        if word_clean in CUSTOM_STOPWORDS: continue
        if target_clean in word_clean: continue 

        # Root-based deduplication
        root = word_clean[:5] if len(word_clean) > 5 else word_clean
        if root in seen_roots: continue
            
        # --- Passed all filters ---
        seen_roots.add(root)
        final_words.append(word_clean) 
        final_scores.append(score)
        
        if len(final_words) >= top_n:
            break
            
    return final_words, final_scores

# ==========================================
# 3. Comparison function
# ==========================================
def analyze_american_self(df, year1, year2, target_word):
    print(f"\nU.S. self-reconstruction: semantic evolution of '{target_word}' ({year1} vs {year2})")
    print("-" * 60)
    
    # Train models
    s1 = df[df['year'] == year1]['final_sentences'].tolist()
    s2 = df[df['year'] == year2]['final_sentences'].tolist()
    
    if len(s1) < 5 or len(s2) < 5: 
        print("Insufficient data; skipping.")
        return

    m1 = Word2Vec(s1, vector_size=300, window=10, min_count=5, seed=42, epochs=20)
    m2 = Word2Vec(s2, vector_size=300, window=10, min_count=5, seed=42, epochs=20)
    
    # Fetch neighbors
    w1, s1_score = get_clean_neighbors(m1, target_word, 10)
    w2, s2_score = get_clean_neighbors(m2, target_word, 10)
    
    # Display results
    res = pd.DataFrame({
        f'{year1} (Pre-Rupture)': w1,
        'Sim 1': [f"{s:.2f}" for s in s1_score],
        '   >>>   ': ['>>>'] * 10,
        f'{year2} (Post-Rupture)': w2,
        'Sim 2': [f"{s:.2f}" for s in s2_score]
    })
    
    display(res)

# ==========================================
# 4. Run analyses
# ==========================================

# 1. U.S. (the state): use 'america' instead of 'united_states'
analyze_american_self(df_text, 2021, 2022, 'america')

# 2. Democracy (the value)
analyze_american_self(df_text, 2021, 2022, 'democracy')

# 3. Freedom (the ideal)
analyze_american_self(df_text, 2021, 2022, 'freedom')

### 8.3 Monthly evidence retrieval: China and Russia/Ukraine/sanctions co-mentions in 2022

In [ ]:
import pandas as pd
import textwrap

# ==========================================
# Core utility: monthly evidence retrieval
# ==========================================
def get_monthly_top_evidence(df, year, target_topic, trigger_words, count_per_month=2):
    print(f"\n{'='*80}")
    print(f"Monthly evidence selection: {year} articles on '{target_topic}' (top {count_per_month} per month)")
    print(f"Logic: find articles where '{target_topic}' and {trigger_words} co-occur in the same sentence")
    print(f"{'='*80}\n")
    
    # 1. Prepare time columns
    if 'pub_date' in df.columns:
        df['pub_date'] = pd.to_datetime(df['pub_date'])
        df['year'] = df['pub_date'].dt.year
        df['month'] = df['pub_date'].dt.month
    
    # 2. Iterate over months 1-12
    for month in range(1, 13):
        # Filter to the current month
        subset = df[(df['year'] == year) & (df['month'] == month)].copy()
        
        if subset.empty:
            continue
            
        # --- 3. Score articles within the month ---
        scored_articles = []
        
        for idx, row in subset.iterrows():
            text = str(row.get('full_text', ''))
            headline = str(row.get('headline', ''))
            
            # Simple sentence split
            sentences = text.replace('Mr.', 'Mr').replace('U.S.', 'US').split('. ')
            
            score = 0
            evidence_sentences = []
            
            # Sentence-by-sentence scan
            for sent in sentences:
                sent_lower = sent.lower()
                
                # Core criterion: target word and trigger word must co-occur in the same sentence
                if target_topic.lower() in sent_lower:
                    hit_triggers = [t for t in trigger_words if t in sent_lower]
                    if hit_triggers:
                        score += 1  # one matching sentence = one point
                        clean_sent = sent.strip().replace('\n', ' ')
                        if len(clean_sent) > 20:
                            evidence_sentences.append(f"[{', '.join(hit_triggers)}] ...{clean_sent}...")
            
            # Only articles with score > 0 (true co-occurrence) qualify
            if score > 0:
                scored_articles.append({
                    'date': row['pub_date'].strftime('%Y-%m-%d'),
                    'headline': headline,
                    'score': score,
                    'evidence': evidence_sentences,
                    'full_text': text
                })
        
        # --- 4. Pick the top articles for the month ---
        # Sort by score (descending)
        scored_articles.sort(key=lambda x: x['score'], reverse=True)
        
        if not scored_articles:
            print(f"\n--- {month}: no articles meeting the co-occurrence criterion ---")
            continue
            
        print(f"\n--- {year}-{month:02d}: {len(scored_articles)} qualifying articles, showing top {count_per_month} ---")
        
        # Display Top N
        for i, article in enumerate(scored_articles[:count_per_month]):
            print(f"\n  [Rank {i+1}] score: {article['score']} | date: {article['date']}")
            print(f"  Headline: {article['headline']}")
            
            # A. Evidence sentences (the core)
            print("  Evidence sentences:")
            for sent in article['evidence'][:3]:  # show top 3 evidence sentences
                print(f"    -> {sent}")
            
            # B. Full text
            print("\n  Full text:")
            print(textwrap.fill(article['full_text'], width=80))
            print("-" * 60)

# ==========================================
# Run retrieval
# ==========================================

# Scenario 1: China and Russia/Ukraine — monthly accomplice-narrative articles
# Criterion: 'China' must co-occur with Ukraine/Sanctions terms in the same sentence
get_monthly_top_evidence(df_text, 2022, 'China', 
                         ['ukraine', 'sanctions', 'russia', 'invasion', 'putin'], 
                         count_per_month=2)

# Scenario 2: Taiwan and war — monthly war-narrative articles
# Criterion: 'Taiwan' must co-occur with Military/Drills terms in the same sentence
get_monthly_top_evidence(df_text, 2022, 'Taiwan', 
                         ['pelosi', 'drills', 'military', 'exercises', 'force'], 
                         count_per_month=2)

### 8.4 Monthly evidence retrieval: weaponization of democratic vocabulary in 2022

In [ ]:
import pandas as pd
import textwrap
import re

# ==========================================
# Core utility: full-text monthly evidence retrieval
# ==========================================
def get_monthly_full_text_evidence(df, year, target_topic_list, trigger_words, count_per_month=2):
    # Compose title
    topic_str = "/".join(target_topic_list)
    print(f"\n{'='*80}")
    print(f"U.S. self-narrative tracking: {year} deep reading on '{topic_str}'")
    print(f"Logic: find articles co-mentioning targets with {trigger_words} in the same sentence")
    print(f"{'='*80}\n")
    
    # 1. Parse pub_date as datetime
    if 'pub_date' in df.columns:
        df['pub_date'] = pd.to_datetime(df['pub_date'])
        df['year'] = df['pub_date'].dt.year
        df['month'] = df['pub_date'].dt.month
    
    # 2. Iterate over months 1-12
    for month in range(1, 13):
        subset = df[(df['year'] == year) & (df['month'] == month)].copy()
        
        if subset.empty:
            continue
            
        scored_articles = []
        
        # 3. Score articles by relevance
        for idx, row in subset.iterrows():
            text = str(row.get('full_text', ''))
            headline = str(row.get('headline', ''))
            
            # Simple sentence split
            sentences = text.replace('Mr.', 'Mr').replace('U.S.', 'US').split('. ')
            
            score = 0
            evidence_sentences = []
            
            for sent in sentences:
                sent_lower = sent.lower()
                
                # A. Check whether any target term is in the sentence
                # Any single match is sufficient
                has_target = any(t.lower() in sent_lower for t in target_topic_list)
                
                if has_target:
                    # B. Check whether any trigger term is also in the sentence
                    hit_triggers = [t for t in trigger_words if t in sent_lower]
                    
                    if hit_triggers:
                        score += 1
                        # Record the evidence sentence
                        clean_sent = sent.strip().replace('\n', ' ')
                        if len(clean_sent) > 20:
                            evidence_sentences.append(f"[{', '.join(hit_triggers)}] ...{clean_sent}...")
            
            if score > 0:
                scored_articles.append({
                    'date': row['pub_date'].strftime('%Y-%m-%d'),
                    'headline': headline,
                    'score': score,
                    'evidence': evidence_sentences,
                    'full_text': text
                })
        
        # 4. Sort and emit top N for the month
        scored_articles.sort(key=lambda x: x['score'], reverse=True)
        
        if not scored_articles:
            print(f"\n--- {year}-{month:02d}: no qualifying articles ---")
            continue
            
        print(f"\n=== {year}-{month:02d}: top {count_per_month} articles ===")
        
        for i, article in enumerate(scored_articles[:count_per_month]):
            print(f"\n  [Rank {i+1}] relevance score: {article['score']} | date: {article['date']}")
            print(f"  Headline: {article['headline']}")
            
            # --- Reading guide: evidence sentences first ---
            print("  Key evidence sentences:")
            for sent in article['evidence'][:3]:  # top 3 evidence sentences
                print(f"    -> {sent}")
            
            # --- Full text ---
            print("\n  Full text:")
            print(textwrap.fill(article['full_text'], width=80))
            print("-" * 80)

# ==========================================
# Run deep reading
# ==========================================

# Scenario 1: Weaponization of democracy
# Logic: find articles where 'democracy' co-occurs with authoritarian/aggression/sovereignty terms
# Purpose: test whether 'democracy' shifts from a domestic concept to a foreign-policy instrument
get_monthly_full_text_evidence(
    df_text, 2022, 
    ['democracy', 'democratic', 'democracies'],  # target terms
    ['authoritarian', 'autocra', 'aggression', 'sovereignty', 'dictator', 'tyranny'],  # trigger terms
    count_per_month=2
)

# Scenario 2: Competition for the Global South (U.S. anxiety)
# Logic: find articles where 'United States' co-occurs with Africa/Latin America/Global South/partners terms
# Purpose: test whether the U.S. is actively competing for allies in the Global South
get_monthly_full_text_evidence(
    df_text, 2022, 
    ['united states', 'america', 'washington', 'biden'],  # target terms
    ['africa', 'latin', 'global south', 'developing countries', 'partners', 'alliances', 'pacific islands'],  # trigger terms
    count_per_month=2
)

# Scenario 3: The cost of freedom
# Logic: find articles where 'freedom' co-occurs with cost/consequence/sacrifice/struggle terms
# Purpose: test whether 'freedom' shifts toward a register of burden and sacrifice
get_monthly_full_text_evidence(
    df_text, 2022, 
    ['freedom', 'liberty', 'free world'],  # target terms
    ['cost', 'price', 'sacrifice', 'consequence', 'fight', 'struggle', 'pain'],  # trigger terms
    count_per_month=2
)